# 資料作成に使えるガロア体計算のテキスト作成プログラム

In [2]:
import unicodedata

In [12]:
# ガロア体の計算util

def generate_galois_tables():
    exp = [0] * 256
    log = [0] * 256
    v = 1
    for i in range(255):
        exp[i] = v
        log[v] = i
        v <<= 1
        if v & 0x100:
            v ^= 0x11D
    exp[255] = exp[0]
    return exp, log

exp_table, log_table = generate_galois_tables()

def gf_mul(x, y):
    """ガロア体 GF(2^8) 上での乗算"""
    if x == 0 or y == 0:
        return 0
    return exp_table[(log_table[x] + log_table[y]) % 255]


def gf_div(x, y):
    if y == 0: raise ZeroDivisionError()
    if x == 0: return 0
    return exp_table[(log_table[x] - log_table[y]) % 255]

print("GF(2^8) の準備が完了しました。")

GF(2^8) の準備が完了しました。


### ガロア体の筆算をテキスト化

In [13]:
# ==========================================
# 2. 筆算出力関数の定義
# ==========================================
def get_display_width(text):
    """全角・半角を考慮した文字列の表示幅を取得"""
    return sum(2 if unicodedata.east_asian_width(c) in 'FWA' else 1 for c in text)

def print_gf_long_division(dividend, divisor):
    """
    ガロア体の多項式除算の筆算をテキスト出力する関数
    dividend: 割られる数 (例: メッセージ多項式)
    divisor:  割る数 (例: 生成多項式)
    """
    current_rem = list(dividend)
    steps = len(dividend) - len(divisor) + 1
    w = 6 # カラムの表示幅
    
    # 割る数（左側）の文字列整形
    gen_str = "".join([f"{g:>{w-1}} " for g in divisor])
    prefix_space = " " * (len(gen_str) + 3)
    
    # 商の計算
    quotients = []
    temp_rem = list(dividend)
    for i in range(steps):
        quotients.append(temp_rem[i])
        if temp_rem[i] != 0:
            for j in range(len(divisor)):
                temp_rem[i+j] ^= gf_mul(temp_rem[i], divisor[j])
                
    # 1段目：商の表示
    q_str = "".join([f"{q:>{w}}" for q in quotients])
    print(f"{prefix_space}{q_str}   (商)")
    print(f"{prefix_space}{'-' * (w * len(dividend) + 2)}")
    
    # 2段目：割る数 ) 割られる数
    div_str = "".join([f"{m:>{w}}" for m in dividend])
    print(f"{gen_str}) {div_str}")
    
    # 各ステップの筆算処理
    for i in range(steps):
        coef = current_rem[i]
        
        # 掛け算して引く数の算出
        sub_row = [gf_mul(coef, g) for g in divisor]
        
        # 左側の注釈
        note = f"({coef}を掛けて引く)"
        indent_spaces = len(gen_str) + 3 + i * w
        
        note_w = get_display_width(note)
        if indent_spaces > note_w + 1:
            left_part = note + " " * (indent_spaces - note_w - 1) + "^"
        else:
            left_part = note + " ^"
            
        sub_str = "".join([f"{s:>{w}}" for s in sub_row])
        print(f"{left_part}{sub_str[1:]}")
        
        # XORの実行
        for j in range(len(divisor)):
            current_rem[i+j] ^= sub_row[j]
            
        # 横線
        dash_indent = " " * indent_spaces
        print(f"{dash_indent}{'-' * (w * len(divisor) + 2)}")
        
        # 次の余りの表示
        if i < steps - 1:
            rem_indent = " " * (len(gen_str) + 3 + (i+1) * w)
            end_idx = min(i + 1 + len(divisor), len(dividend))
            rem_str = "".join([f"{current_rem[k]:>{w}}" for k in range(i+1, end_idx)])
            print(f"{rem_indent}{rem_str}")
        else:
            rem_indent = " " * (len(gen_str) + 3 + (i+1) * w)
            rem_str = "".join([f"{current_rem[k]:>{w}}" for k in range(i+1, len(dividend))])
            print(f"{rem_indent}{rem_str}   (余り)")




In [14]:
# ==========================================
# 3. 実行
# ==========================================
# メッセージ多項式 (データ部 + 誤り訂正語)
msg = [72, 65, 99, 97, 162, 112, 246]
# 生成多項式
gen = [1, 15, 54, 120, 64]

print_gf_long_division(msg, gen)

                                     72    65    99   (商)
                                 --------------------------------------------
    1    15    54   120    64 )     72    65    99    97   162   112   246
(72を掛けて引く)                ^   72   159   172   140   247
                                 --------------------------------
                                          222   207   237    85   112
(222を掛けて引く)                     ^  222   158    93   132   153
                                       --------------------------------
                                                 81   176   209   233   246
(81を掛けて引く)                            ^   81    24   112   192   249
                                             --------------------------------
                                                      168   161    41    15   (余り)


In [15]:
# ==========================================
# 2. 余りからシンドロームの計算
# ==========================================
# 筆算（生成多項式での割り算）で得られた余り
# 多項式表現: 168x^3 + 161x^2 + 41x + 15
remainder = [168, 161, 41, 15]

# 計算するシンドロームの数（誤り訂正語の数と同じ）
num_ec_codewords = 4
syndromes = []

print("【余り多項式への代入によるシンドローム計算】")
print(f"対象の余り: {remainder}\n")

for i in range(num_ec_codewords):
    alpha_i = exp_table[i] # 代入する値 (α^0, α^1, α^2, α^3)
    
    syndrome = 0
    # ホーナー法を用いた多項式の代入計算
    for coef in remainder:
        # 現在の値を α^i 倍して、次の係数をXORで足す
        syndrome = coef ^ gf_mul(syndrome, alpha_i)
        
    syndromes.append(syndrome)
    print(f"x = α^{i} ({alpha_i:<3}) を代入 => S_{i} = {syndrome}")

print(f"\n▼ 最終的に得られたシンドローム ▼")
print(syndromes)

【余り多項式への代入によるシンドローム計算】
対象の余り: [168, 161, 41, 15]

x = α^0 (1  ) を代入 => S_0 = 47
x = α^1 (2  ) を代入 => S_1 = 202
x = α^2 (4  ) を代入 => S_2 = 60
x = α^3 (8  ) を代入 => S_3 = 231

▼ 最終的に得られたシンドローム ▼
[47, 202, 60, 231]


### BM法手順可視化

In [16]:
# BM法のトレースプログラム
syndromes = [47, 202, 60, 231]

# 初期化
Lambda = [1]
B = [1]
L = 0
m = 1

print("【初期状態】")
print(f"Lambda(x): {Lambda}, B(x): {B}, L: {L}, m: {m}\n")

for k in range(len(syndromes)):
    print(f"--- ステップ {k} (S_{k} = {syndromes[k]} の読み込み) ---")
    
    # 1. d の計算
    d = syndromes[k]
    for i in range(1, L + 1):
        if i < len(Lambda):
            d ^= gf_mul(Lambda[i], syndromes[k - i])
    print(f"  不一致度 d = {d}")
    
    if d == 0:
        print("  予測成功: Lambda(x) は変更せず、m を +1 します。")
        m += 1
    else:
        print("  予測失敗: Lambda(x) を修正します。")
        
        # T(x) = Lambda(x) - d * x^m * B(x) の計算
        shift_B = [0] * m + B
        T = list(Lambda)
        # リストのサイズ合わせ
        if len(shift_B) > len(T):
            T.extend([0] * (len(shift_B) - len(T)))
            
        for i in range(len(shift_B)):
            T[i] ^= gf_mul(d, shift_B[i])
            
        # L の更新判定
        if 2 * L <= k:
            print(f"  2L <= k ({2*L} <= {k}) が真のため、L と B(x) を更新します。")
            # B(x) = Lambda(x) / d
            B = [gf_div(coef, d) for coef in Lambda]
            L = k + 1 - L
            m = 1
        else:
            print(f"  2L <= k ({2*L} <= {k}) が偽のため、L と B(x) は維持します。")
            m += 1
            
        Lambda = T
        
    # 末尾の余分な0を削除して見やすくする
    while len(Lambda) > 1 and Lambda[-1] == 0:
        Lambda.pop()
        
    print(f"  [更新後] Lambda(x): {Lambda}, L: {L}, m: {m}\n")

print(f"最終的な誤り位置多項式 Lambda(x): {Lambda}")

【初期状態】
Lambda(x): [1], B(x): [1], L: 0, m: 1

--- ステップ 0 (S_0 = 47 の読み込み) ---
  不一致度 d = 47
  予測失敗: Lambda(x) を修正します。
  2L <= k (0 <= 0) が真のため、L と B(x) を更新します。
  [更新後] Lambda(x): [1, 47], L: 1, m: 1

--- ステップ 1 (S_1 = 202 の読み込み) ---
  不一致度 d = 235
  予測失敗: Lambda(x) を修正します。
  2L <= k (2 <= 1) が偽のため、L と B(x) は維持します。
  [更新後] Lambda(x): [1, 16], L: 1, m: 2

--- ステップ 2 (S_2 = 60 の読み込み) ---
  不一致度 d = 0
  予測成功: Lambda(x) は変更せず、m を +1 します。
  [更新後] Lambda(x): [1, 16], L: 1, m: 3

--- ステップ 3 (S_3 = 231 の読み込み) ---
  不一致度 d = 0
  予測成功: Lambda(x) は変更せず、m を +1 します。
  [更新後] Lambda(x): [1, 16], L: 1, m: 4

最終的な誤り位置多項式 Lambda(x): [1, 16]


### チェン探索

In [19]:
# --- チェン探索 (Chien Search) の実行 ---
Lambda = [1, 16] # 導出された誤り位置多項式

print("【チェン探索の計算過程】")
print(f"対象の方程式: Λ(x) = {Lambda[0]} + {Lambda[1]}x \n") # + {Lambda[2]}x^2

roots = []
error_positions = []

for i in range(255):
    x = exp_table[i] # 代入する値 (α^i)
    
    # 各項の計算
    term0 = Lambda[0]
    term1 = gf_mul(Lambda[1], x)
    
    # x^2 以上があるときは α^(2i) として計算
    #x2 = exp_table[(i * 2) % 255]
    #term2 = gf_mul(Lambda[2], x2)
    
    # 全てをXORで足し合わせる
    result = term0 ^ term1 #^ term2
    
    # 出力のフォーマット（最初の3回と、解が見つかった時だけ出力）
    if i < 3 or result == 0:
        print(f"x = α^{i:<3} ({x:<3}) を代入: {term0:<3} ⊕ {term1:<3} = {result:<3}", end="")
        
        if result == 0:
            print("  <= 解を発見！")
            roots.append(i)
            # 物理的な位置（後ろからのインデックス）に変換
            error_positions.append(255 - i)
        else:
            print("  (ハズレ)")
    elif i == 3:
        print("    ...")
        print("    (中略：ひたすら総当たりを続ける)")
        print("    ...")

print("\n【探索結果と物理位置への変換】")
for root, pos in zip(roots, error_positions):
    print(f"・見つかった解: α^{root}")
    print(f"  => 逆元の指数: 255 - {root} = {pos}")
    print(f"  => エラーの位置は、メッセージの末尾から {pos} 乗の位置\n")

【チェン探索の計算過程】
対象の方程式: Λ(x) = 1 + 16x 

x = α^0   (1  ) を代入: 1   ⊕ 16  = 17   (ハズレ)
x = α^1   (2  ) を代入: 1   ⊕ 32  = 33   (ハズレ)
x = α^2   (4  ) を代入: 1   ⊕ 64  = 65   (ハズレ)
    ...
    (中略：ひたすら総当たりを続ける)
    ...
x = α^251 (216) を代入: 1   ⊕ 1   = 0    <= 解を発見！

【探索結果と物理位置への変換】
・見つかった解: α^251
  => 逆元の指数: 255 - 251 = 4
  => エラーの位置は、メッセージの末尾から 4 乗の位置



### フォルニー法

In [20]:
# ガロア体 GF(2^8) の準備 (これまでの関数と同じ)
def generate_galois_tables():
    exp = [0] * 256
    log = [0] * 256
    v = 1
    for i in range(255):
        exp[i] = v
        log[v] = i
        v <<= 1
        if v & 0x100: v ^= 0x11D
    exp[255] = exp[0]
    return exp, log

exp_table, log_table = generate_galois_tables()

def gf_mul(x, y):
    if x == 0 or y == 0: return 0
    return exp_table[(log_table[x] + log_table[y]) % 255]

def gf_div(x, y):
    if y == 0: raise ZeroDivisionError()
    if x == 0: return 0
    return exp_table[(log_table[x] - log_table[y]) % 255]

# --- フォルニー法による復元検証 ---

# 受信した壊れたメッセージ
received_message = [72, 65, 99, 97, 162, 112, 246]

# 前段までに判明している情報
S0 = 47                  # シンドローム S_0
X1_exp = 4               # エラー位置 (後ろから4番目)
X1 = exp_table[X1_exp]   # エラー位置のガロア体要素 (α^4 = 16)
Lambda_prime = 16        # Λ'(x) の値

print("【フォルニー法によるエラー値の計算】")
# Omega(x) の評価値 (今回は定数項 S0 となる)
omega_val = S0
print(f"・分子の評価: Ω(X_1^-1) = {omega_val}")
print(f"・分母の評価: Λ'(X_1^-1) = {Lambda_prime}")

# 公式: e1 = X1 * Omega(X1^-1) / Lambda'(X1^-1)
numerator = gf_mul(X1, omega_val)
e1 = gf_div(numerator, Lambda_prime)

print(f"・計算されたエラー値 (e_1): {e1}\n")

print("【データの修復】")
# メッセージ長は7、後ろから4乗の位置はインデックス換算で (7 - 1) - 4 = 2
error_index = 2
broken_value = received_message[error_index]

print(f"・該当インデックス ({error_index}番目) の壊れたデータ: {broken_value}")

# XORによる修復
repaired_value = broken_value ^ e1
print(f"・修復計算: {broken_value} ⊕ {e1} = {repaired_value}")

# メッセージの書き換え
received_message[error_index] = repaired_value
print(f"\n▼ 最終的な修復メッセージ ▼")
print(received_message)
print("完璧に元通りになりました！")

【フォルニー法によるエラー値の計算】
・分子の評価: Ω(X_1^-1) = 47
・分母の評価: Λ'(X_1^-1) = 16
・計算されたエラー値 (e_1): 47

【データの修復】
・該当インデックス (2番目) の壊れたデータ: 99
・修復計算: 99 ⊕ 47 = 76

▼ 最終的な修復メッセージ ▼
[72, 65, 76, 97, 162, 112, 246]
完璧に元通りになりました！
